# Testing Notebook — Prediction Request ke TF Serving

This notebook is used to test the Adult Income model deployed using TensorFlow Serving.

**Note:** The model accepts **preprocessed numeric data** (not raw strings).
Preprocessing (z-score, label encoding) was already done in pandas in the main pipeline.

---

## Prasyarat

Make sure TF Serving is running:

```bash
# Via Docker (direkomendasikan)
docker run -p 8501:8501 \
  --mount type=bind,source=$(pwd)/serving_model/adult_income_model,target=/models/adult_income \
  -e MODEL_NAME=adult_income \
  -t tensorflow/serving
```

Make sure the `data/test_samples.json` file is available (generated by the main pipeline).

## 1. Import Library

In [1]:
import requests
import json
import os
import tensorflow as tf
import base64

SERVING_URL = 'http://localhost:8501/v1/models/adult_income:predict'
DATA_ROOT   = os.path.join(os.getcwd(), 'data')
print('Serving URL:', SERVING_URL)
print('Data root  :', DATA_ROOT)

2026-05-14 06:03:42.996986: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-14 06:03:43.572609: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-05-14 06:03:43.572637: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-05-14 06:03:43.684918: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 06:03:45.393661: W tensorflow/stream_executor/platform/de

Serving URL: http://localhost:8501/v1/models/adult_income:predict
Data root  : /home/sintiawati_putu04/mlops-tfx-submission/data


## 2. Cek Status Model Serving

In [3]:
status_url = 'http://localhost:8501/v1/models/adult_income'
response = requests.get(status_url)
print('Status code:', response.status_code)
print('Model status:')
print(json.dumps(response.json(), indent=2))

Status code: 200
Model status:
{
  "model_version_status": [
    {
      "version": "1778737948",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## 3. Load Test Samples (Preprocessed Data)

Load samples that were preprocessed by the main pipeline. This data already has:
- Z-score normalization for numerical features
- Label encoding for categorical features
- Label (income) dropped (not sent to the model)

In [4]:
samples_path = os.path.join(DATA_ROOT, 'test_samples.json')
with open(samples_path, 'r') as f:
    test_samples = json.load(f)

print(f'Loaded {len(test_samples)} test samples')
for i, s in enumerate(test_samples):
    print(f'\nSample {i+1}:')
    print(f'  age={s["age"]:.2f}, workclass={s["workclass"]}, education-num={s["education-num"]:.2f}')
    print(f'  hours-per-week={s["hours-per-week"]:.2f}, sex={s["sex"]}, race={s["race"]}')

Loaded 3 test samples

Sample 1:
  age=0.03, workclass=0, education-num=1.13
  hours-per-week=-0.08, sex=0, race=0

Sample 2:
  age=0.87, workclass=1, education-num=1.13
  hours-per-week=-2.33, sex=0, race=0

Sample 3:
  age=-0.04, workclass=2, education-num=-0.44
  hours-per-week=-0.08, sex=0, race=0


## 4. Kirim Prediction Request

Model menggunakan signature `serving_default` yang menerima serialized tf.Example.
Kita buat tf.Example dari data numerik preprocessed, lalu kirim via REST API.

In [5]:
def sample_to_b64(sample):
    """Convert preprocessed sample dict to base64-encoded serialized tf.Example."""
    feature = {}
    for key, val in sample.items():
        if isinstance(val, float):
            feature[key] = tf.train.Feature(float_list=tf.train.FloatList(value=[val]))
        elif isinstance(val, int):
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[val]))
        else:
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[int(val)]))
    tf_example = tf.train.Example(features=tf.train.Features(feature=feature))
    return base64.b64encode(tf_example.SerializeToString()).decode('utf-8')


def predict(examples, url=SERVING_URL):
    payload = {
        'instances': [{'examples': {'b64': ex}} for ex in examples]
    }
    response = requests.post(url, json=payload)
    if response.status_code != 200:
        raise Exception(f'Error {response.status_code}: {response.text}')
    return response.json()['predictions']


# Kirim semua test samples
examples = [sample_to_b64(s) for s in test_samples]
predictions = predict(examples)

print('=== Prediction Results ===')
for i, pred in enumerate(predictions, 1):
    prob = pred[0]
    label = '>50K' if prob >= 0.5 else '<=50K'
    print(f'Sample {i}: p(>50K)={prob:.4f} -> {label}')

=== Prediction Results ===
Sample 1: p(>50K)=0.0626 -> <=50K
Sample 2: p(>50K)=0.2378 -> <=50K
Sample 3: p(>50K)=0.0114 -> <=50K


## 5. Create Manual Samples for Testing

To test the model with custom data, we need to provide data in **preprocessed numeric** format.
Columns to send (without the `income` label):

In [6]:
FEATURE_COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
]

print(f'Model accepts {len(FEATURE_COLUMNS)} features:')
for i, col in enumerate(FEATURE_COLUMNS):
    dtype = 'float64' if col in ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week'] else 'int64'
    print(f'  {i+1:2d}. {col:<18} ({dtype})')

Model accepts 14 features:
   1. age                (float64)
   2. workclass          (int64)
   3. fnlwgt             (float64)
   4. education          (int64)
   5. education-num      (float64)
   6. marital-status     (int64)
   7. occupation         (int64)
   8. relationship       (int64)
   9. race               (int64)
  10. sex                (int64)
  11. capital-gain       (float64)
  12. capital-loss       (float64)
  13. hours-per-week     (float64)
  14. native-country     (int64)


### 5.1 High-Income Sample (>50K)

Create a sample deliberately made to resemble high-income characteristics
(mature age, higher education, many working hours).

In [7]:
# Sample: likely >50K
high_income = {
    'age': 2.5,              # z-score: ~2.5 std dev above mean
    'workclass': 4,           # Private
    'fnlwgt': -0.5,           # relatively low fnlwgt
    'education': 9,           # Masters
    'education-num': 2.0,     # high z-score
    'marital-status': 2,      # Married-civ-spouse
    'occupation': 4,          # Exec-managerial
    'relationship': 0,        # Husband
    'race': 4,                # White
    'sex': 1,                 # Male
    'capital-gain': 3.0,      # high z-score (large capital gain)
    'capital-loss': -0.2,     # near 0
    'hours-per-week': 2.0,    # high z-score (>50 hrs/week)
    'native-country': 39,     # United-States
}

# Sample: likely <=50K
low_income = {
    'age': -1.2,              # z-score: relatively young
    'workclass': 4,           # Private
    'fnlwgt': 0.5,            # average fnlwgt
    'education': 2,           # HS-grad
    'education-num': -1.0,    # low z-score
    'marital-status': 0,      # Divorced
    'occupation': 8,          # Other-service
    'relationship': 3,        # Not-in-family
    'race': 4,                # White
    'sex': 0,                 # Female
    'capital-gain': -0.2,    # near 0
    'capital-loss': -0.2,    # near 0
    'hours-per-week': -0.5,  # below average
    'native-country': 39,     # United-States
}

manual_samples = [high_income, low_income]
manual_examples = [sample_to_b64(s) for s in manual_samples]
manual_predictions = predict(manual_examples)

print('=== Manual Samples Prediction ===')
for i, (desc, pred) in enumerate(zip(['High-Income (>50K)', 'Low-Income (<=50K)'], manual_predictions), 1):
    prob = pred[0]
    label = '>50K' if prob >= 0.5 else '<=50K'
    print(f'{i}. {desc}: p(>50K)={prob:.4f} -> Predicted: {label}')

=== Manual Samples Prediction ===
1. High-Income (>50K): p(>50K)=0.9997 -> Predicted: >50K
2. Low-Income (<=50K): p(>50K)=0.0000 -> Predicted: <=50K


## 6. Batch Prediction

Send multiple samples at once in a single request.

In [8]:
batch_examples = [sample_to_b64(s) for s in test_samples] + manual_examples
batch_predictions = predict(batch_examples)

print(f'Total {len(batch_predictions)} predictions')
print('-' * 60)
print(f'{"#":<4} {"Type":<20} {"Prob(>50K)":<12} {"Prediction"}')
print('-' * 60)
types = ['Test Sample'] * len(test_samples) + ['Manual High'] + ['Manual Low']
for i, (t, pred) in enumerate(zip(types, batch_predictions), 1):
    prob = pred[0]
    label = '>50K' if prob >= 0.5 else '<=50K'
    print(f'{i:<4} {t:<20} {prob:<12.4f} {label}')

Total 5 predictions
------------------------------------------------------------
#    Type                 Prob(>50K)   Prediction
------------------------------------------------------------
1    Test Sample          0.0626       <=50K
2    Test Sample          0.2378       <=50K
3    Test Sample          0.0114       <=50K
4    Manual High          0.9997       >50K
5    Manual Low           0.0000       <=50K


## 7. Conclusion

Adult Income Classification model successfully:
1. ✅ Deployed using TensorFlow Serving on port 8501
2. ✅ Accepts serialized `tf.Example` input with preprocessed numeric data
3. ✅ Returns probability prediction for income `>50K`
4. ✅ Works correctly for both single and batch predictions

Decision threshold: probability ≥ 0.5 → `>50K`, probability < 0.5 → `<=50K`